# 1.1.11 Advanced Python Concepts

Metaclasses, descriptors, __slots__, generator send(), dataclasses, itertools — ML framework internals.

In [ ]:
# Metaclass: Model Registry (like HuggingFace AutoModel)
class ModelRegistry(type):
    _registry = {}
    def __new__(mcs, name, bases, ns, **kw):
        cls = super().__new__(mcs, name, bases, ns)
        if bases: mcs._registry[kw.get('tag', name.lower())] = cls
        return cls
    @classmethod
    def build(mcs, tag, **kw): return mcs._registry[tag](**kw)

class BaseModel(metaclass=ModelRegistry): pass
class Linear(BaseModel, tag='linear'):
    def __init__(self, lr=0.01): self.lr=lr
    def __repr__(self): return f'Linear(lr={self.lr})'
class Tree(BaseModel, tag='tree'):
    def __init__(self, depth=3): self.depth=depth
    def __repr__(self): return f'Tree(depth={self.depth})'

print('Registered:', list(ModelRegistry._registry.keys()))
m = ModelRegistry.build('linear', lr=0.005)
print('Built:', m)

In [ ]:
# Descriptors for validated HyperParams
class BoundedFloat:
    def __init__(self, lo, hi): self.lo=lo; self.hi=hi; self.name=''
    def __set_name__(self, owner, name): self.name=name
    def __get__(self, obj, t=None): return obj.__dict__.get(self.name) if obj else self
    def __set__(self, obj, v):
        v = float(v)
        if not (self.lo <= v <= self.hi): raise ValueError(f'{self.name} out of [{self.lo},{self.hi}]')
        obj.__dict__[self.name] = v

class HP:
    lr = BoundedFloat(1e-6, 1.0)
    dropout = BoundedFloat(0.0, 0.9)
    def __init__(self, lr=1e-3, dropout=0.1): self.lr=lr; self.dropout=dropout
    def __repr__(self): return f'HP(lr={self.lr}, dropout={self.dropout})'

hp = HP(lr=0.001, dropout=0.2); print(hp)
try: hp.lr = 5.0
except ValueError as e: print('Caught:', e)

In [ ]:
# Generator with send() — running mean coroutine
def running_mean():
    total=0.0; count=0
    x = yield 0.0
    while True:
        total+=x; count+=1
        x = yield total/count

gen = running_mean(); next(gen)
for loss in [0.85, 0.72, 0.61, 0.55, 0.50]:
    mean = gen.send(loss)
    print(f'loss={loss:.2f}  mean={mean:.4f}')

In [ ]:
# Dataclasses with __post_init__
from dataclasses import dataclass, field
import hashlib, time
@dataclass
class Experiment:
    name: str; model: str; epochs: int=10
    tags: list = field(default_factory=list)
    run_id: str = field(init=False)
    def __post_init__(self):
        self.run_id = hashlib.md5(f'{self.name}{time.time()}'.encode()).hexdigest()[:8]
    def summary(self): return f'[{self.run_id}] {self.name} | {self.model} | epochs={self.epochs}'

for e in [Experiment('baseline','linear',50,['v1']),Experiment('deep','tree',100,['v2'])]:
    print(e.summary())

In [ ]:
# itertools for ML
import itertools
grid = list(itertools.product([0.1,0.01,0.001],[32,64]))
print('Grid search:', grid)
print('islice 5:', list(itertools.islice(iter(range(1000)), 5)))